In [ ]:
from google.cloud import bigquery
import os
import plotly.express as px
import pandas as pd
from pyproj import Transformer

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../data/auth/sante-et-territoires-89f4496cab7e.json" 
client = bigquery.Client() 

query = """
SELECT *
FROM `sante-et-territoires.finess.equipements_sociaux_clean`
"""
df = client.query(query).to_dataframe()




In [ ]:
df.info()

In [ ]:
df = df.drop_duplicates()


In [ ]:

# Création du transformateur Lambert 93 -> WGS84
transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)

# Conversion
df["longitude"], df["latitude"] = transformer.transform(
    df["coordxet"].values,df["coordyet"].values
)

# Aperçu
print(df[["nofinesset", "coordxet", "coordyet", "latitude", "longitude"]].head())

In [ ]:
df.head()

In [ ]:
df['lib_categorie_etablissement'].value_counts()

In [ ]:
# Regroupement des catégories de types d'établissements
def regrouper_categorie(cat):
    cat = cat.lower()

    if any(x in cat for x in ["hospital", "clinique", "soins", "dialyse", "ssr", "had", "cancer"]):
        return "Hopitaux cliniques"

    if any(x in cat for x in ["handicap", "ime", "itep", "mas", "fam", "esat", "sensoriel", "e.s.a.t.", "i.m.e.", "i.t.e.p.", "m.a.s.", "s.a.v.s.", "foyer de vie", "entreprise adaptée", "esat", "institut d'éducation motrice", "service mandataire judiciaire à la protection des majeurs", "c.a.m.s.p."]):
        return "Médico-social handicap"

    if any(x in cat for x in ["ehpad", "ehpa", "autonomie", "personnes âgées", "longue durée", "personnes agées"]):
        return "Personnes âgées"

    if any(x in cat for x in ["chrs", "casa", "c.a.d.a", "hébergement", "foyer", "maison relais"]):
        return "Social / Hébergement"

    if any(x in cat for x in ["pharmacie", "centre de santé", "maison de santé", "laboratoire", "c.m.p.", "cabinet", "infirmier", "kinésithérapeute", "dentiste", "opticien", "structure dispensatrice à domicile d'oxygène à usage médical", "maison médicale de garde (MMG)"]):
        return "Soins de ville"

    if any(x in cat for x in ['prévention', "vaccination", "dépistage", "csapa", "caarud" ]):
        return "Prévention / Santé publique"

    if any(x in cat for x in ["enfance", "camsp", "aemo", "aed", "pouponnière", "maison d'enfants", "educative", "protection maternelle et infantile", "protection infantile", "pmi", "p.m.i."]):
        return "Enfance / Protection"

    if any(x in cat for x in ["cpts", "mdph", "coordination", "groupement"]):
        return "Coordination / Administration"
    if any(x in cat for x in ["centre d'accueil", "accueil"]):
            return "Centres d'accueil"
    if any(x in cat for x in ["ecoles"]):
        return "Ecoles"
    return "Autres"

# Application
df["groupe"] = df["lib_categorie_etablissement"].apply(regrouper_categorie)

In [ ]:
df['libde_equipement'].value_counts()

In [55]:
import unicodedata

def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8")
    s = s.replace("’", "'")
    s = " ".join(s.split())
    return s.lower()

def regrouper_categorie(cat):
    cat = clean_text(cat)

    if any(x in cat for x in [
        "ehpad", "personnes agees", "hebergement pour personnes agees", "maison de retraite", "hbergement pour personnes agees dependantes"
    ]):
        return "Personnes âgées"

    if any(x in cat for x in [
        "ime", "itep", "sessad", "enfance", "enfant", "educatif", "pedagogique", "cmpp", "scolarisation", "educative", "c.m.p.p."
    ]):
        return "Enfance"

    if any(x in cat for x in [
        "handicap", "handicape", "handicapes", "handicapees", "adapte"
    ]):
        return "Handicap"

    if any(x in cat for x in [
        "soins a domicile", "aide a domicile", "service a domicile", "a domicile"
    ]):
        return "Aide et soins à domicile"

    if any(x in cat for x in [
        "psychologique", "psy", "psychiatrie", "cmpp"
    ]):
        return "Aide psychologique"

    if any(x in cat for x in [
        "hebergement", "maison relais", "pension de famille", "maisons relais", "accueil temporaire"
    ]):
        return "Hébergement"

    if any(x in cat for x in [
        "social", "clico", "service social", "intervention educative", "club prevention"
    ]):
        return "Social"

    if any(x in cat for x in [
        "reinsertion", "adaptation", "rehabilitation", "protection des majeurs"
    ]):
        return "Réinsertion et adaptation"
    
    if any(x in cat for x in [
        "aidant", "aidants"
    ]):
        return "Aidants"
    if any(x in cat for x in [
        "cure"
    ]):
        return "Cure"

    return "Autres"


# Application
df["groupe_equipement"] = df["libde_equipement"].apply(regrouper_categorie)

In [56]:
df['groupe_equipement'].value_counts()

groupe_equipement
Personnes âgées              25545
Aide et soins à domicile     23869
Enfance                      19949
Handicap                     18544
Hébergement                   7906
Autres                        3694
Social                        3402
Cure                          1670
Réinsertion et adaptation      688
Aidants                        463
Aide psychologique              19
Name: count, dtype: int64

In [57]:
df[df["groupe_equipement"] == "Autres"]["libde_equipement"].value_counts()


libde_equipement
Accueil orientation soins accompagnement diff spécifiques       1192
Préparation à la vie professionnelle                             480
Activ.Club et Équipe de Prévention                               319
Information,conseil, expertise, coordination                     293
Tutelle curatelle mandat spécial sauvegarde justice pers maj     273
Mesure d'accompagnement judiciaire                               176
Observation Orientation Pour Mineurs Justice                     154
Observation en Milieu Ouvert Pour Mineurs Justice (O.M.O)        123
Mesure judiciaire aide gestion budget familial                   116
Consultation d'Orientation Pour Mineurs Justice                  114
Activité Serv. Travailleuses Familiales                           83
Equipe mobile santé précarité                                     65
Regroupement des Calculs (Annexes 24)                             56
C.H.R.S. Hors les murs                                            52
Evaluat réentraîn

In [58]:
#nettoyage des doublons
df_clean = df.drop_duplicates()

In [59]:
#je crée un colonne code_insee à partir de la colonne code commune et de la colonne departement
df_clean.loc[:, "code_insee"] = (
    df_clean["departement"].astype(str).str.zfill(2)
    + df_clean["commune"].astype(str).str.zfill(3)
)

df_clean.head()

,nofinessej,nofinesset,libde_equipement,libta_equipement,lib_categorie_etablissement,coordyet,coordxet,raison_sociale,raison_sociale_longue,capinstot,commune,departement,libdepartement,client,longitude,latitude,groupe,groupe_equipement,code_insee
0,920002268,920805066,Hébergement ouvert en foyer pour adultes handi...,Hébergement de Nuit Eclaté,Appartement Thérapeutique,6864353.6,643674.5,APPARTEMENTS THERAPEUTIQUES,APPARTEMENTS THERAPEUTIQUES DE SURESNES ET PUT...,12,073,92,HAUTS DE SEINE,Tous Types de Déficiences Pers.Handicap.(sans ...,2.232039,48.876739,Autres,Handicap,92073
1,750045072,010010874,Hébergement médico soc personnes en difficulté...,Hébergement de Nuit Eclaté,Appartement de Coordination Thérapeutique (A.C...,6568991.9,870430.3,ACT BASILIADE DE BOURG-EN-BRESSE,ACT BASILIADE DE BOURG-EN-BRESSE,21,053,01,AIN,Personnes nécessitant prise en charge psycho s...,5.210436,46.199300,Coordination / Administration,Hébergement,01053
2,750045072,010010874,Accueil orientation soins accompagnement diff ...,Prestation en milieu ordinaire,Appartement de Coordination Thérapeutique (A.C...,6568991.9,870430.3,ACT BASILIADE DE BOURG-EN-BRESSE,ACT BASILIADE DE BOURG-EN-BRESSE,15,053,01,AIN,Personnes nécessitant prise en charge psycho s...,5.210436,46.199300,Coordination / Administration,Autres,01053
3,750045072,010010874,Hébergement médico soc personnes en difficulté...,Hébergement Complet Internat,Appartement de Coordination Thérapeutique (A.C...,6568991.9,870430.3,ACT BASILIADE DE BOURG-EN-BRESSE,ACT BASILIADE DE BOURG-EN-BRESSE,4,053,01,AIN,Personnes nécessitant prise en charge psycho s...,5.210436,46.199300,Coordination / Administration,Hébergement,01053
4,780020715,020015392,Hébergement médico soc personnes en difficulté...,Hébergement Complet Internat,Appartement de Coordination Thérapeutique (A.C...,6906351.9,706455.2,ACT CENTRE HENRI VINCENT,ACT CENTRE HENRI VINCENT - FONDATION DIACONESS...,18,810,02,AISNE,Personnes nécessitant prise en charge psycho s...,3.088658,49.256794,Coordination / Administration,Hébergement,02810


In [60]:
# latitudes longitudes en float
df_clean.loc[:,'latitude'] = df_clean['latitude'].astype(float)
df_clean.loc[:,'longitude'] = df_clean['longitude'].astype(float)

In [61]:
#je filtre les données sur les départements de la région Occitanie
departements_occitanie = ["09","11", "12", "30","31", "32", "34", "46", "48","65", "66", "81", "82",  ]
df_occitanie = df_clean[df_clean["departement"].isin(departements_occitanie)]

In [ ]:
df_occitanie.head()

In [ ]:
df_occitanie.info()

In [62]:
#j'exporte en csv dans les data de l'application

df_occitanie.to_csv("../src/data/equipements_occitanie.csv", index=False, sep=";", encoding="utf-8")


In [ ]:
df_occitanie['nofinesset'].value_counts()

In [ ]:
# vérifier les capacités
df_occitanie=pd.read_csv("../src/data/equipements_occitanie.csv", sep=";", encoding="utf-8")

In [ ]:
df_occitanie.head()

In [ ]:
df_occitanie['capinstot'].value_counts()

In [ ]:
df_occitanie.isnull().sum()


In [ ]:
df_occitanie_cap_nulle = df_occitanie[df_occitanie['capinstot'].isnull()]

In [ ]:
df_occitanie['groupe_equipement'].value_counts()

In [ ]:
(df_occitanie['capinstot']== 0.0).sum()

In [ ]:
len(df_occitanie)

In [ ]:
df_occitanie_cap_nulle['groupe_equipement'].value_counts()

In [ ]:
#pourcentage de valeur manquantes
len(df_occitanie_cap_nulle)/len(df_occitanie)*100
